# Clase 6 — Implementación y gobierno: el modelo que no está documentado no existe
## Modelador de Riesgo de Crédito · Academia Bayes

**Demo en vivo con el dataset A — Banco Austral** (datos 100% sintéticos)

Cinco clases construyendo evidencia. Hoy convertimos ese notebook en cuatro
cosas que sobreviven a que usted se vaya de vacaciones: un **artefacto**, un
**contrato**, un **expediente** y un **gobierno**.

| # | Sección | ~min |
|---|---|---|
| 1 | Las clases 1–5 en celdas de referencia | 4 |
| 2 | En producción NO existe DEV: congelar el binning | 5 |
| 3 | El artefacto de despliegue: ~5.500 caracteres de JSON | 5 |
| 4 | La prueba de paridad (y la identidad del caso) | 5 |
| 5 | Los dos bugs del binning: el que se ve y el que no | 6 |
| 6 | Contrato de datos: qué bloquea y qué solo avisa | 6 |
| 7 | La corrida de producción, sellada de crudo a score | 6 |
| 8 | Romper el trail de dos maneras · el sello externo | 5 |
| 9 | Lineage: los cuatro eslabones con su hash | 3 |
| 10 | El model card, generado desde la corrida | 5 |
| 11 | El resumen ejecutivo de una página | 3 |
| 12 | El atajo: `governance`, `audit` y `report` de `nikodym` | 7 |

> 🎯 **El principio de hoy**: producción no CALCULA, produccion APLICA. Todo lo
> que el score necesita tiene que estar dentro de un archivo versionado — y
> todo lo que se ejecutó tiene que quedar registrado con su hash.

In [1]:
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (9, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

BASE = "https://raw.githubusercontent.com/nexolabs-gh/datos-riesgo-credito/main"
PREFIJO = "austral"   # dataset A: Banco Austral

clientes       = pd.read_parquet(f"{BASE}/{PREFIJO}_clientes.parquet")
solicitudes    = pd.read_parquet(f"{BASE}/{PREFIJO}_solicitudes.parquet")
comportamiento = pd.read_parquet(f"{BASE}/{PREFIJO}_comportamiento.parquet")
bureau         = pd.read_parquet(f"{BASE}/{PREFIJO}_bureau.parquet")

print(f"solicitudes:    {solicitudes.shape[0]:,} filas × {solicitudes.shape[1]} columnas")
print(f"comportamiento: {comportamiento.shape[0]:,} filas × {comportamiento.shape[1]} columnas")

solicitudes:    32,301 filas × 10 columnas
comportamiento: 905,014 filas × 13 columnas


### Recordatorio: el diccionario de datos, a mano

Seguimos con las mismas cuatro tablas de Banco Austral, así que dejamos el
diccionario cargado otra vez. Cada variable que fabriquemos hoy nace de una
columna de estas tablas: si no sabes qué mide la columna, no puedes defender la
variable frente a un comité.

[`diccionario_austral.md`](https://github.com/nexolabs-gh/datos-riesgo-credito/blob/main/diccionario_austral.md) ·
la celda siguiente lo muestra completo y deja `mostrar_diccionario(...)` definida.

In [2]:
import urllib.request

from IPython.display import Markdown, display

URL_DICCIONARIO = f"{BASE}/diccionario_{PREFIJO}.md"
_DICCIONARIO = urllib.request.urlopen(URL_DICCIONARIO).read().decode("utf-8")


def mostrar_diccionario(tabla=None):
    """Muestra el diccionario de datos oficial; sin argumento, lo muestra completo."""
    if tabla is None:
        display(Markdown(_DICCIONARIO))
        return
    secciones = _DICCIONARIO.split("\n## ")[1:]
    elegidas = [s for s in secciones if s.lower().startswith(tabla.lower())]
    if not elegidas:
        disponibles = [s.split("(")[0].strip() for s in secciones]
        raise ValueError(f"No existe la tabla {tabla!r}. Disponibles: {disponibles}")
    display(Markdown("## " + elegidas[0]))


mostrar_diccionario()

<IPython.core.display.Markdown object>


---
## 1. Las clases 1–5, en celdas de referencia

Las cinco celdas siguientes son la receta acumulada del curso — **las mismas
del notebook de la clase 5**, sin un carácter de diferencia: target y muestras
(C1), fábrica de variables (C2), binning/WoE (C2–C3), el modelo y su scorecard
(C3), y la calibración con la master scale (C4).

La sexta recupera de la clase 5 los indicadores que hoy van a la **ficha del
modelo**: Gini y KS por muestra, PSI del score contra la bandeja TTD, y el
backtesting binomial global. (El tablero completo de nueve indicadores es el
de la clase pasada; hoy solo lo citamos.)

> Se ejecutan, no se narran. Si alguna falla, el problema está antes de hoy.

In [3]:
# ---------- clase 1: target, población y muestras ----------
mora_w = comportamiento.pivot_table(index="id_cliente", columns="mes",
                                    values="dias_mora", aggfunc="first")
meses = list(mora_w.columns)
pos_mes = {m: k for k, m in enumerate(meses)}
M = mora_w.to_numpy()
fila_de = {c: k for k, c in enumerate(mora_w.index)}

cursadas = solicitudes[solicitudes["aprobada"]].copy()
cursadas["_t"] = cursadas["fecha_solicitud"].map(pos_mes)
cursadas["_i"] = cursadas["id_cliente"].map(fila_de)

peor = np.full(len(cursadas), np.nan)
for k, (i, t0) in enumerate(zip(cursadas["_i"], cursadas["_t"])):
    if t0 + 12 <= len(meses) - 1:
        peor[k] = M[i, t0 + 1: t0 + 13].max()
cursadas["peor_dpd_12m"] = peor
cursadas["malo"] = np.where(np.isnan(peor), np.nan, (peor >= 90).astype(float))
cursadas["indeterminado"] = (peor >= 30) & (peor < 90)

antig = clientes.set_index("id_cliente")["fecha_alta_cliente"].map(
    lambda s: (pd.Period(s, freq="M") - pd.Period(meses[0], freq="M")).n)

periodo = cursadas[cursadas["fecha_solicitud"].between("2024-07", "2025-06")]
periodo = periodo.sort_values("fecha_solicitud", kind="stable").drop_duplicates(
    "id_cliente", keep="first").copy()
periodo["antiguedad_meses"] = periodo["_t"] - periodo["id_cliente"].map(antig)
base = periodo[periodo["malo"].notna() & ~periodo["marca_fraude"]
               & (periodo["antiguedad_meses"] >= 6)].copy()

rng = np.random.default_rng(42)
base["muestra"] = np.where(base["fecha_solicitud"] <= "2025-02",
                           np.where(rng.random(len(base)) < 0.7, "DEV", "HO"), "OOT")
ttd_pob = cursadas[cursadas["fecha_solicitud"] >= "2025-07"].copy()
ttd_pob["antiguedad_meses"] = ttd_pob["_t"] - ttd_pob["id_cliente"].map(antig)
ttd_pob = ttd_pob[ttd_pob["antiguedad_meses"] >= 6]
ttd_pob["muestra"] = "TTD"
poblacion = pd.concat([base, ttd_pob], ignore_index=True)
print(poblacion["muestra"].value_counts().reindex(["DEV", "HO", "OOT", "TTD"]).to_string())

muestra
DEV    3501
HO     1465
OOT    2114
TTD    8585


In [4]:
# ---------- clase 2: panel ancho + fábrica declarativa + ratios ----------
def pivotar(df, columna):
    ancho = df.pivot_table(index="id_cliente", columns="mes", values=columna, aggfunc="first")
    return ancho.reindex(index=mora_w.index, columns=meses).to_numpy()

PANEL = {c: pivotar(comportamiento, c) for c in
         ["dias_mora", "saldo_linea", "cupo_linea", "saldo_tc", "cupo_tc", "saldo_consumo",
          "monto_pagado", "monto_facturado", "abonos_cuenta", "saldo_ahorro", "n_productos"]}
PANEL.update({c: pivotar(bureau, c) for c in
              ["deuda_otras_inst", "n_otras_inst", "consultas_mes", "peor_mora_sistema"]})
with np.errstate(divide="ignore", invalid="ignore"):
    PANEL["uso_linea"] = np.where(PANEL["cupo_linea"] > 0,
                                  PANEL["saldo_linea"] / PANEL["cupo_linea"], np.nan)
    PANEL["uso_tc"] = np.where(PANEL["cupo_tc"] > 0,
                               PANEL["saldo_tc"] / PANEL["cupo_tc"], np.nan)
PANEL["deuda_interna"] = PANEL["saldo_linea"] + PANEL["saldo_tc"] + PANEL["saldo_consumo"]
PANEL["deuda_total"] = PANEL["deuda_interna"] + PANEL["deuda_otras_inst"]
PANEL["en_mora"] = np.where(np.isnan(PANEL["dias_mora"]), np.nan,
                            (PANEL["dias_mora"] > 0).astype(float))
PANEL["en_mora_sistema"] = np.where(np.isnan(PANEL["peor_mora_sistema"]), np.nan,
                                    (PANEL["peor_mora_sistema"] > 0).astype(float))

def prom(V):  return np.nanmean(V, axis=1)
def vmax(V):  return np.nanmax(V, axis=1)
def suma(V):  return np.nansum(V, axis=1)
def delta(V): return V[:, -1] / np.maximum(V[:, 0], 1) - 1
def recencia(V):
    evento = np.nan_to_num(V, nan=0.0) > 0
    k = evento.shape[1]
    ultimo = k - 1 - np.argmax(evento[:, ::-1], axis=1)
    return np.where(evento.any(axis=1), k - ultimo, k + 1)
AGREGADORES = {"prom": prom, "max": vmax, "suma": suma, "delta": delta, "recencia": recencia}

def agregar_ventana(matriz, filas, t0s, k, funcion):
    salida = np.full(len(filas), np.nan)
    for t0 in np.unique(t0s):
        sel = t0s == t0
        salida[sel] = funcion(matriz[filas[sel], max(t0 - k, 0):t0])
    return salida

FABRICA = [
    ("dias_mora", "max", (3, 6, 12), "dias_mora_max_{k}m"),
    ("dias_mora", "prom", (3, 6, 12), "dias_mora_prom_{k}m"),
    ("en_mora", "suma", (3, 6, 12), "n_meses_mora_{k}m"),
    ("en_mora", "recencia", (12,), "meses_desde_mora_12m"),
    ("uso_linea", "prom", (3, 6, 12), "uso_linea_prom_{k}m"),
    ("uso_linea", "max", (3, 6, 12), "uso_linea_max_{k}m"),
    ("uso_tc", "prom", (3, 6, 12), "uso_tc_prom_{k}m"),
    ("uso_tc", "max", (3, 6, 12), "uso_tc_max_{k}m"),
    ("saldo_consumo", "prom", (3, 6, 12), "saldo_consumo_prom_{k}m"),
    ("saldo_consumo", "delta", (6, 12), "saldo_consumo_delta_{k}m"),
    ("deuda_interna", "prom", (3, 6, 12), "deuda_interna_prom_{k}m"),
    ("deuda_interna", "max", (3, 6, 12), "deuda_interna_max_{k}m"),
    ("deuda_total", "prom", (3, 6, 12), "deuda_total_prom_{k}m"),
    ("deuda_total", "delta", (6, 12), "deuda_total_delta_{k}m"),
    ("monto_pagado", "suma", (3, 6, 12), "pagos_{k}m"),
    ("monto_facturado", "suma", (3, 6, 12), "facturacion_{k}m"),
    ("abonos_cuenta", "prom", (3, 6, 12), "abonos_prom_{k}m"),
    ("abonos_cuenta", "delta", (6, 12), "abonos_delta_{k}m"),
    ("saldo_ahorro", "prom", (3, 6, 12), "saldo_ahorro_prom_{k}m"),
    ("n_productos", "prom", (3, 6, 12), "n_productos_prom_{k}m"),
    ("deuda_otras_inst", "prom", (3, 6, 12), "deuda_otras_prom_{k}m"),
    ("deuda_otras_inst", "max", (3, 6, 12), "deuda_otras_max_{k}m"),
    ("deuda_otras_inst", "delta", (6,), "delta_deuda_otras_{k}m"),
    ("n_otras_inst", "prom", (3, 6, 12), "n_otras_inst_prom_{k}m"),
    ("peor_mora_sistema", "max", (3, 6, 12), "peor_mora_sistema_{k}m"),
    ("en_mora_sistema", "suma", (3, 6, 12), "n_meses_mora_sistema_{k}m"),
    ("consultas_mes", "suma", (3, 6, 12), "consultas_{k}m"),
    ("consultas_mes", "recencia", (12,), "meses_desde_consulta_12m"),
]
META = ["id_solicitud", "id_cliente", "fecha_solicitud", "_t", "_i",
        "muestra", "malo", "indeterminado", "peor_dpd_12m"]

filas_p, t0s_p = poblacion["_i"].to_numpy(), poblacion["_t"].to_numpy()
X = poblacion[META + ["antiguedad_meses", "monto_solicitado", "plazo_meses",
                      "destino", "canal"]].copy()
for serie, agg, ventanas, plantilla in FABRICA:
    for k in ventanas:
        X[plantilla.format(k=k)] = agregar_ventana(PANEL[serie], filas_p, t0s_p, k,
                                                   AGREGADORES[agg])
t_ult = np.maximum(t0s_p - 1, 0)
for serie in ["dias_mora", "saldo_linea", "cupo_linea", "saldo_tc", "cupo_tc",
              "saldo_consumo", "saldo_ahorro", "deuda_otras_inst", "n_productos",
              "peor_mora_sistema", "uso_linea", "uso_tc"]:
    X[f"{serie}_ult"] = PANEL[serie][filas_p, t_ult]

ingreso = X["abonos_prom_6m"].to_numpy()
deuda_ult = (X["saldo_linea_ult"] + X["saldo_tc_ult"]
             + X["saldo_consumo_ult"] + X["deuda_otras_inst_ult"])
X["carga_financiera"] = deuda_ult / np.maximum(ingreso, 1)
X["pago_sobre_fact_6m"] = X["pagos_6m"] / np.maximum(X["facturacion_6m"], 1)
X["pago_sobre_fact_12m"] = X["pagos_12m"] / np.maximum(X["facturacion_12m"], 1)
X["deuda_int_sobre_sistema"] = (X["saldo_linea_ult"] + X["saldo_tc_ult"]
                                + X["saldo_consumo_ult"]) / np.maximum(deuda_ult, 1)
X["ahorro_sobre_ingreso"] = X["saldo_ahorro_ult"] / np.maximum(ingreso, 1)
X["monto_sobre_ingreso"] = X["monto_solicitado"] / np.maximum(ingreso, 1)
X["cuota_est_sobre_ingreso"] = (X["monto_solicitado"] / X["plazo_meses"]) / np.maximum(ingreso, 1)
X = X.merge(clientes[["id_cliente", "edad", "sexo", "region", "estado_civil",
                      "nivel_educacional", "tipo_empleo", "renta_liquida",
                      "n_dependientes"]], on="id_cliente", how="left")
X["renta_vs_abonos"] = X["renta_liquida"] / np.maximum(ingreso, 1)

predictoras = [c for c in X.columns if c not in META]
filtro = X["malo"].notna() & ~X["indeterminado"]
dev = X[(X["muestra"] == "DEV") & filtro]
ho  = X[(X["muestra"] == "HO") & filtro]
oot = X[(X["muestra"] == "OOT") & filtro]
ttd = X[X["muestra"] == "TTD"]
print(f"MATRIZ: {len(X):,} filas × {len(predictoras)} candidatas")
print(f"DEV={len(dev):,} ({dev['malo'].mean():.1%} malos) · HO={len(ho):,} · "
      f"OOT={len(oot):,} · TTD={len(ttd):,}")

MATRIZ: 15,665 filas × 108 candidatas
DEV=3,322 (5.0% malos) · HO=1,397 · OOT=2,004 · TTD=8,585


In [5]:
# ---------- clase 2/3: binning con `ref` + tabla WoE ----------
def binear(x, bins=5, umbral_moda=0.35, ref=None):
    """Binning del curso. Con `ref`, los cortes se calculan en `ref` y se aplican a `x`."""
    x = pd.Series(x).reset_index(drop=True)
    base_b = x if ref is None else pd.Series(ref).reset_index(drop=True)
    if base_b.dtype == object or isinstance(base_b.dtype, pd.CategoricalDtype):
        return x.astype(str).where(x.notna(), "MISSING"), None
    if base_b.nunique(dropna=True) <= bins:
        return x.astype(str).where(x.notna(), "MISSING"), None
    moda = base_b.mode().iloc[0]
    frac_moda = (base_b == moda).mean()
    resto = base_b[base_b != moda] if frac_moda > umbral_moda else base_b
    cortes = np.unique(np.nanquantile(resto.dropna(), np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return x.astype(str).where(x.notna(), "MISSING"), None
    cortes[0], cortes[-1] = -np.inf, np.inf
    categorias = pd.cut(x, cortes)
    orden = [str(c) for c in categorias.cat.categories]
    etiquetas = categorias.astype(str)
    if frac_moda > umbral_moda:
        etiquetas = etiquetas.where(x != moda, f"= {moda:.4g}")
        orden = [f"= {moda:.4g}"] + orden
    return etiquetas.where(x.notna(), "MISSING"), orden + ["MISSING"]


def tabla_woe(x, y, bins=5, ref=None):
    """Tabla WoE de una variable y su IV (convención: target 1 = malo)."""
    etiquetas, orden = binear(x, bins, ref=ref)
    tab = pd.DataFrame({"bin": etiquetas.values, "y": pd.Series(y).values}) \
            .groupby("bin")["y"].agg(["count", "sum"])
    tab.columns = ["n", "malos"]
    if orden:
        tab = tab.reindex([o for o in orden if o in tab.index])
    tab["buenos"] = tab["n"] - tab["malos"]
    tab["tasa_malos"] = tab["malos"] / tab["n"]
    p_malos  = (tab["malos"] + 0.5) / (tab["malos"].sum() + 0.5 * len(tab))
    p_buenos = (tab["buenos"] + 0.5) / (tab["buenos"].sum() + 0.5 * len(tab))
    tab["woe"] = np.log(p_buenos / p_malos)
    tab["iv_aporte"] = (p_buenos - p_malos) * tab["woe"]
    return tab, float(tab["iv_aporte"].sum())

In [6]:
# ---------- clase 3 comprimida: el modelo final y el scorecard ----------
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

ELEGIDAS = ["uso_linea_prom_12m", "meses_desde_mora_12m", "uso_tc_prom_12m",
            "deuda_interna_max_3m", "antiguedad_meses", "deuda_otras_prom_12m",
            "carga_financiera", "uso_tc_prom_3m"]

MAPAS = {v: tabla_woe(dev[v], dev["malo"])[0]["woe"] for v in ELEGIDAS}

def a_woe(df, variables):
    """Transforma `df` al WoE de DEV: binea con cortes de DEV y mapea su WoE."""
    F = pd.DataFrame(index=range(len(df)))
    for v in variables:
        etiquetas, _ = binear(df[v], ref=dev[v])
        F[v] = etiquetas.map(MAPAS[v]).fillna(0.0).values
    return F

modelo = sm.Logit(dev["malo"].values,
                  sm.add_constant(a_woe(dev, ELEGIDAS), has_constant="add")
                  ).fit(disp=0, maxiter=200)
assert (modelo.params.drop("const") < 0).all(), "signos: revisar antes de seguir"

PDO, SCORE_BASE, ODDS_BASE = 20, 600, 50
factor = PDO / np.log(2)
offset = SCORE_BASE - factor * np.log(ODDS_BASE)

def pd_de(df):
    """PD cruda del modelo (WoE y cortes de DEV, como producción)."""
    return np.asarray(modelo.predict(sm.add_constant(a_woe(df, ELEGIDAS),
                                                     has_constant="add")))

def score_desde_pd(p):
    """Score PDO desde una PD, con clip bilateral (evita ±inf en los bordes)."""
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    return offset + factor * np.log((1 - p) / p)

print(f"modelo de la clase 3 re-ajustado: {len(ELEGIDAS)} variables · "
      f"Gini DEV {2 * roc_auc_score(dev['malo'], pd_de(dev)) - 1:.3f}")

modelo de la clase 3 re-ajustado: 8 variables · Gini DEV 0.757


In [7]:
# ---------- clase 4 comprimida: calibración a la TC + master scale ----------
from scipy.optimize import brentq
from scipy.special import expit, logit

MUESTRAS = {"DEV": dev, "HO": ho, "OOT": oot, "TTD": ttd}
PD_RAW = {m: pd_de(df) for m, df in MUESTRAS.items()}

gb = X[X["malo"].notna() & ~X["indeterminado"]]
TC = float(gb.groupby("fecha_solicitud")["malo"].mean().mean())   # promedio simple

lp_dev = logit(np.clip(PD_RAW["DEV"], 1e-12, 1 - 1e-12))
delta = float(brentq(lambda d: expit(lp_dev + d).mean() - TC, -5, 5, xtol=1e-12))

PD_CAL, SCORE_CAL = {}, {}
for m in MUESTRAS:
    lp = logit(np.clip(PD_RAW[m], 1e-12, 1 - 1e-12))
    PD_CAL[m] = expit(lp + delta)
    SCORE_CAL[m] = score_desde_pd(PD_CAL[m])
assert abs(PD_CAL["DEV"].mean() - TC) < 1e-9

CORTES_BANDA = [540, 560, 580, 600, 620, 640, 660]
ETIQUETAS = ["E", "D", "C2", "C1", "B2", "B1", "A2", "A1"]      # peor → mejor

def banda_de(score):
    # right=False: el corte cae en la banda superior (convención de C4)
    return pd.cut(score, [-np.inf] + CORTES_BANDA + [np.inf], labels=ETIQUETAS,
                  right=False)

print(f"TC {TC:.4f} · delta {delta:+.4f} · el modelo en producción está listo: "
      f"PD_CAL y SCORE_CAL por muestra, bandas de la master scale definidas")

TC 0.0542 · delta +0.1115 · el modelo en producción está listo: PD_CAL y SCORE_CAL por muestra, bandas de la master scale definidas


In [8]:
# ---------- clase 5 comprimida: los indicadores que van a la ficha ----------
from sklearn.metrics import roc_auc_score, roc_curve

def gini_ks(pd_pred, y):
    auc = roc_auc_score(y, pd_pred)
    fpr, tpr, _ = roc_curve(y, pd_pred)
    return 2 * auc - 1, float((tpr - fpr).max())

# MUESTRAS ya viene filtrada por la celda de C4 (sin indeterminados ni malo NaN):
# usar X[X["muestra"]==m] daría otro n y las métricas no cuadrarían con C5
metricas = {}
for m in ("DEV", "HO", "OOT"):
    g, k = gini_ks(PD_CAL[m], MUESTRAS[m]["malo"].values)
    metricas[m] = {"gini": g, "ks": k}
metricas = pd.DataFrame(metricas).T
metricas["caida_gini_rel"] = 1 - metricas["gini"] / metricas.loc["DEV", "gini"]

def psi_bandas(esperado, actual, suavizado=1e-4):
    e = pd.Series(banda_de(esperado)).value_counts(normalize=True)
    a = pd.Series(banda_de(actual)).value_counts(normalize=True)
    cats = ETIQUETAS
    pe = e.reindex(cats).fillna(0) + suavizado
    pa = a.reindex(cats).fillna(0) + suavizado
    return float(((pa - pe) * np.log(pa / pe)).sum())

psi_ttd = psi_bandas(SCORE_CAL["DEV"], SCORE_CAL["TTD"])

from scipy import stats
binom_global = stats.binomtest(int(oot["malo"].sum()), len(oot),
                               float(PD_CAL["OOT"].mean()))

print(metricas.round(4).to_string())
print(f"\nPSI del score DEV→TTD: {psi_ttd:.4f}")
print(f"Backtesting binomial global (OOT): p = {binom_global.pvalue:.3f}")
print("\nTablero completo de la clase 5: 4 🟢 · 5 🟡 · 0 🔴")

       gini      ks  caida_gini_rel
DEV  0.7570  0.5965          0.0000
HO   0.6981  0.5532          0.0778
OOT  0.6750  0.5454          0.1083

PSI del score DEV→TTD: 0.0130
Backtesting binomial global (OOT): p = 0.562

Tablero completo de la clase 5: 4 🟢 · 5 🟡 · 0 🔴


---
## 2. En producción NO existe DEV

Miren la firma de la función que veníamos usando para binear:

```python
binear(x, ref=dev[v])
```

Ese `ref=dev[v]` recalcula los cortes de cada bin **mirando la muestra de
desarrollo**. En el notebook es correcto. En el servidor de scoring es
imposible: DEV son 3.322 filas de 2024 que el motor de originación no tiene,
no debe tener y no va a tener.

La misma trampa vive en todo lo que se "ajusta": los WoE, el δ de calibración,
el factor/offset del PDO, los cortes de la master scale.

> **Regla:** si producción tiene que CALCULAR algo que dependa de la población,
> ya está mal. Producción solo APLICA.

Así que el primer paso es **congelar**: convertir la partición en datos.

In [9]:
def es_numerica(s):
    """Numérica de verdad: el `StringDtype` de pandas NO lo es."""
    return pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

def clave_discreta(v):
    """0 y 0.0 son EL MISMO bin.

    Sin esta normalización, un int64 en DEV y un float64 en producción (basta
    que aparezca un NaN para que pandas promueva la columna) generan claves
    distintas — "0" vs "0.0" — y la variable entera se neutraliza en silencio.
    """
    if pd.isna(v):
        return "MISSING"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return str(int(v)) if float(v).is_integer() else repr(float(v))
    return str(v)

def woe_normalizado(mapa, spec):
    """Reescribe las claves del WoE con la convención del bin congelado.

    En una variable discreta de tipo float, `binear` etiqueta "0.0" y el bin
    congelado "0": es la MISMA partición con otro nombre. Si el mapa queda a
    medio camino, la variable se neutraliza en silencio. (En el dataset A no
    pasa; en el B sí — y es el tipo de detalle que revienta en producción.)
    """
    if spec["tipo"] != "discreto":
        return {str(b): float(w) for b, w in mapa.items()}
    salida = {}
    for b, w in mapa.items():
        if str(b) == "MISSING":
            salida["MISSING"] = float(w)
            continue
        try:
            salida[clave_discreta(float(b))] = float(w)
        except (TypeError, ValueError):
            salida[str(b)] = float(w)
    return salida

def misma_particion(a, b):
    """¿Dos etiquetados agrupan los mismos casos, aunque se llamen distinto?"""
    ca = pd.factorize(pd.Series(a).astype(str))[0]
    cb = pd.factorize(pd.Series(b).astype(str))[0]
    t = pd.crosstab(ca, cb).astype(bool)
    return bool(t.sum(axis=1).max() == 1 and t.sum(axis=0).max() == 1)

def ajustar_bins(base, bins=5, umbral_moda=0.35):
    """Misma partición que `binear(x, ref=base)`, pero como DATOS."""
    base = pd.Series(base).reset_index(drop=True)
    if not es_numerica(base):
        return {"tipo": "categorico"}
    if base.nunique(dropna=True) <= bins:
        return {"tipo": "discreto",
                "valores": sorted({clave_discreta(v) for v in base.dropna()})}
    moda = base.mode().iloc[0]
    frac = float((base == moda).mean())
    resto = base[base != moda] if frac > umbral_moda else base
    cortes = np.unique(np.nanquantile(resto.dropna(), np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return {"tipo": "discreto",
                "valores": sorted({clave_discreta(v) for v in base.dropna()})}
    # los ±inf se guardan como None: `Infinity` no es JSON válido
    cortes = [None] + [float(c) for c in cortes[1:-1]] + [None]
    spec = {"tipo": "numerico", "cortes": cortes, "moda_aparte": frac > umbral_moda}
    if spec["moda_aparte"]:
        spec["moda"] = int(moda) if float(moda).is_integer() else float(moda)
        spec["etiqueta_moda"] = f"= {moda:.4g}"
    return spec

def aplicar_bins(x, spec):
    """Asigna el bin con la spec congelada. PRESERVA el índice (la identidad)."""
    x = pd.Series(x)
    if spec["tipo"] == "categorico":
        return x.astype(str).where(x.notna(), "MISSING")
    if spec["tipo"] == "discreto":
        return pd.Series([clave_discreta(v) for v in x], index=x.index)
    interiores = [float(c) for c in spec["cortes"][1:-1]]
    et = pd.Series(pd.cut(x.values, np.array([-np.inf, *interiores, np.inf])).astype(str),
                   index=x.index)
    if spec["moda_aparte"]:
        et = et.where(x != spec["moda"], spec["etiqueta_moda"])
    return et.where(x.notna(), "MISSING")

spec_demo = ajustar_bins(dev[ELEGIDAS[0]])
print(f"spec congelada de `{ELEGIDAS[0]}`:")
print(json.dumps(spec_demo, indent=1, ensure_ascii=False))

spec congelada de `uso_linea_prom_12m`:
{
 "tipo": "numerico",
 "cortes": [
  null,
  0.21489151515151517,
  0.2599440476190476,
  0.3074674388674389,
  0.36350317066446103,
  null
 ],
 "moda_aparte": false
}


### La prueba que hay que hacer SIEMPRE: ¿congelar cambió algo?

Congelar no sirve de nada si la partición congelada no es **idéntica** a la de
desarrollo. Se verifica bin a bin, en las cuatro muestras — no "en promedio",
no "casi todas": idéntica.

In [10]:
iguales, renombradas = {}, set()
for v in ELEGIDAS:
    spec = ajustar_bins(dev[v])
    for m, sub in [("DEV", dev), ("HO", ho), ("OOT", oot), ("TTD", ttd)]:
        congelado = aplicar_bins(sub[v], spec)
        desarrollo = binear(sub[v], ref=dev[v])[0]
        if (congelado.values == desarrollo.values).all():
            iguales[(v, m)] = True
        else:
            # etiqueta distinta, misma partición (discreta float: "0.0" vs "0")
            iguales[(v, m)] = misma_particion(congelado, desarrollo)
            if iguales[(v, m)]:
                renombradas.add(v)

n_ok = sum(iguales.values())
print(f"comparaciones idénticas: {n_ok} de {len(iguales)} "
      f"({len(ELEGIDAS)} variables × 4 muestras)")
assert n_ok == len(iguales), "el binning congelado cambia la PARTICIÓN"
print("✅ la partición congelada es la misma, sin usar DEV en el camino"
      + (f" ({sorted(renombradas)}: etiqueta normalizada)" if renombradas else ""))

comparaciones idénticas: 32 de 32 (8 variables × 4 muestras)
✅ la partición congelada es la misma, sin usar DEV en el camino


---
## 3. El artefacto de despliegue

Esto es el modelo. Todo lo demás — el notebook, las slides, el informe — es
documentación de cómo llegamos a él.

Regla de oro: **si algo no está aquí, producción no lo tiene.**

In [11]:
CUTOFF = 560   # la decisión de la clase 4

def contrato_datos(dev, variables, tolerancia_missing=3.0):
    """Qué puede llegar: tipo, rango y missing tolerado, por variable."""
    clausulas = {}
    for v in variables:
        s = dev[v]
        tope = min(1.0, float(s.isna().mean()) * tolerancia_missing + 0.01)
        if not es_numerica(s):
            clausulas[v] = {"tipo": "categorico",
                            "categorias": sorted(s.dropna().astype(str).unique().tolist()),
                            "pct_missing_dev": float(s.isna().mean()),
                            "pct_missing_max": tope}
        else:
            clausulas[v] = {"tipo": "numerico",
                            "minimo": float(np.nanmin(s.values)),
                            "maximo": float(np.nanmax(s.values)),
                            "pct_missing_dev": float(s.isna().mean()),
                            "pct_missing_max": tope}
    return clausulas

artefacto = {
    "modelo_id": "austral-scorecard-consumo",
    "cartera_prefijo": "austral",
    "cartera": "Crédito de consumo — Banco Austral",
    "version": "1.0.0",
    "fecha_construccion": "2026-09-10",
    "variables": list(ELEGIDAS),
    "binning": {v: ajustar_bins(dev[v]) for v in ELEGIDAS},
    "woe": {v: woe_normalizado(MAPAS[v], ajustar_bins(dev[v])) for v in ELEGIDAS},
    "coeficientes": {"const": float(modelo.params["const"]),
                     **{v: float(modelo.params[v]) for v in ELEGIDAS}},
    "escalado": {"pdo": 20, "score_base": 600, "odds_base": 50,
                 "factor": float(factor), "offset": float(offset)},
    "calibracion": {"tendencia_central": float(TC), "delta": float(delta),
                    "metodo": "brentq: media de PD ajustada en DEV = TC"},
    "master_scale": {"cortes": CORTES_BANDA, "etiquetas": ETIQUETAS},
    "politica": {"cutoff": CUTOFF, "regla": "aprobar si score >= cutoff"},
    "contrato_datos": contrato_datos(dev, ELEGIDAS),
    "woe_faltante": 0.0,
}

texto = json.dumps(artefacto, ensure_ascii=False, sort_keys=True, allow_nan=False)
print(f"artefacto: {len(texto):,} caracteres de JSON")
print(f"  {len(artefacto['variables'])} variables · "
      f"{sum(len(m) for m in artefacto['woe'].values())} bins con WoE")
print(f"  δ = {artefacto['calibracion']['delta']:+.4f} · "
      f"cutoff {artefacto['politica']['cutoff']}")

artefacto: 5,463 caracteres de JSON
  8 variables · 41 bins con WoE
  δ = +0.1115 · cutoff 560


> ℹ️ `allow_nan=False` no es un detalle de estilo: obliga a que el artefacto
> sea **JSON estándar**. Un archivo con `NaN` o `Infinity` adentro lo lee
> Python y lo rechaza casi cualquier otro sistema — y el motor de scoring de un
> banco rara vez es Python.
>
> Lo mismo con `pickle`: un artefacto que necesita su versión exacta de
> scikit-learn para abrirse no es un artefacto, es una dependencia disfrazada.

In [12]:
# el hash del artefacto: la identidad de ESTE modelo, no de "el scorecard"
def hash_texto(t):
    return hashlib.sha256(t.encode("utf-8")).hexdigest()

def hash_json(obj):
    # sort_keys=True es lo que hace el hash comparable: sin él, el mismo
    # contenido en otro orden daría otro hash y el lineage sería ruido
    return hash_texto(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str))

def hash_df(df):
    esquema = [{"pos": i, "nombre": str(c), "dtype": str(df[c].dtype)}
               for i, c in enumerate(df.columns)]
    h = hashlib.sha256()
    h.update(json.dumps(esquema, sort_keys=True, ensure_ascii=False).encode("utf-8"))
    h.update(pd.util.hash_pandas_object(df, index=True).values.tobytes())
    return h.hexdigest()

HASH_ARTEFACTO = hash_json(artefacto)
print(f"hash del artefacto: {HASH_ARTEFACTO}")

hash del artefacto: 98845d693591d141112674264774a76b7088d1b8d73fb7cc626cbb71f8171094


---
## 4. El motor de scoring, y la prueba de paridad

`puntuar` es todo lo que corre en el servidor: no toca DEV, no reajusta nada,
no mira el target. Fíjense en dos detalles que parecen menores y no lo son:

1. **El índice viaja con el score.** Si el motor devuelve una columna suelta,
   el merge posterior le puede asignar el score a otro cliente — y ese error no
   lo detecta ningún test de métricas.
2. **Los casos con un bin que DEV nunca vio salen marcados.** Un WoE neutro no
   es una estimación del riesgo: es declarar "no sé". Esos casos van a revisión
   manual, no a aprobación automática.

In [13]:
from scipy.special import expit, logit

def score_desde_art(p, art):
    """Score PDO usando SOLO los parámetros del artefacto."""
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    return art["escalado"]["offset"] - art["escalado"]["factor"] * logit(p)

def bandas_de(score, art):
    return pd.cut(score, [-np.inf] + list(art["master_scale"]["cortes"]) + [np.inf],
                  labels=art["master_scale"]["etiquetas"], right=False)

def puntuar(df, art):
    """PD → PD calibrada → score → banda → decisión, SOLO con el artefacto."""
    lp = np.full(len(df), art["coeficientes"]["const"], dtype=float)
    sin_mapa = np.zeros(len(df), dtype=int)
    for v in art["variables"]:
        et = aplicar_bins(df[v], art["binning"][v])
        crudo = et.map(art["woe"][v]).astype(float)
        falta = crudo.isna().values
        sin_mapa += falta.astype(int)
        lp += art["coeficientes"][v] * np.where(falta, art["woe_faltante"], crudo.values)
    pd_cruda = expit(lp)
    pd_c = expit(logit(np.clip(pd_cruda, 1e-12, 1 - 1e-12)) + art["calibracion"]["delta"])
    sc = score_desde_art(pd_c, art)
    salida = pd.DataFrame({"pd_calibrada": pd_c, "score": sc, "banda": bandas_de(sc, art),
                           "bins_sin_mapa": sin_mapa,
                           "apto_automatico": sin_mapa == 0}, index=df.index)
    salida["decision"] = np.where(
        ~salida["apto_automatico"], "revisar",
        np.where(salida["score"] >= art["politica"]["cutoff"], "aprobar", "rechazar"))
    return salida

salida_ttd = puntuar(ttd, artefacto)
print(salida_ttd.head().to_string())
print(f"\níndice preservado: {list(salida_ttd.index[:3])} (id_solicitud, no 0/1/2)")

      pd_calibrada       score banda  bins_sin_mapa  apto_automatico decision
7080      0.003641  649.048671    A2              0             True  aprobar
7081      0.005145  639.027976    B1              0             True  aprobar
7082      0.009709  620.570910    B1              0             True  aprobar
7083      0.013821  610.261348    B2              0             True  aprobar
7084      0.063625  564.711269    C2              0             True  aprobar

índice preservado: [7080, 7081, 7082] (id_solicitud, no 0/1/2)


### La prueba de paridad: lo que TI le va a exigir

Antes de que el artefacto entre a producción, alguien tiene que demostrar que
produce el **mismo número** que el notebook. No "parecido": el mismo.

In [14]:
filas = []
for m in ("DEV", "HO", "OOT", "TTD"):
    sub = {"DEV": dev, "HO": ho, "OOT": oot, "TTD": ttd}[m]
    s = puntuar(sub, artefacto)
    filas.append({"muestra": m, "n": len(sub),
                  "max_dif_score": float(np.abs(s["score"].values - SCORE_CAL[m]).max()),
                  "max_dif_pd": float(np.abs(s["pd_calibrada"].values - PD_CAL[m]).max()),
                  "bandas_iguales": bool((s["banda"].astype(str).values ==
                                          pd.Series(banda_de(SCORE_CAL[m]))
                                          .astype(str).values).all())})
paridad = pd.DataFrame(filas).set_index("muestra")
print(paridad.to_string())
assert paridad["max_dif_score"].max() < 1e-9 and paridad["bandas_iguales"].all()
print("\n✅ paridad exacta: la diferencia es error de punto flotante, no de lógica")

            n  max_dif_score    max_dif_pd  bandas_iguales
muestra                                                   
DEV      3322   1.136868e-13  2.220446e-16            True
HO       1397   1.136868e-13  1.110223e-16            True
OOT      2004   1.136868e-13  1.110223e-16            True
TTD      8585   1.136868e-13  1.665335e-16            True

✅ paridad exacta: la diferencia es error de punto flotante, no de lógica


In [15]:
# Segunda prueba: el score de un caso NO puede depender de con quién vino en
# el lote. Si dependiera, dos clientes idénticos tendrían score distinto según
# el día que postularon — que es exactamente el bug de la sección 5.
caso = ttd.iloc[[7]]
solo = puntuar(caso, artefacto)["score"].iloc[0]
en_lote_10 = puntuar(ttd.iloc[:10], artefacto)["score"].iloc[7]
en_lote_completo = salida_ttd["score"].iloc[7]
print(f"caso #7 · solo: {solo:.4f} · en 10: {en_lote_10:.4f} · "
      f"en {len(ttd):,}: {en_lote_completo:.4f}")
assert solo == en_lote_10 == en_lote_completo
print("✅ el scoring es independiente del lote (no hay estado escondido)")

caso #7 · solo: 632.1990 · en 10: 632.1990 · en 8,585: 632.1990
✅ el scoring es independiente del lote (no hay estado escondido)


---
## 5. Los dos bugs del binning: el que se ve y el que no

Un pipeline de producción que "re-entrena el binner" en cada corrida suena
razonable — hasta se siente responsable. Es el error de implementación más caro
de la industria, y se puede equivocar de dos maneras muy distintas.

**Bug A — el silencioso.** Recalcula los cortes sobre el lote del día y mapea
el WoE por posición (el bin más bajo del lote recibe el WoE del bin más bajo de
desarrollo). Todas las etiquetas encuentran WoE: no hay excepción, no hay valor
nulo, no hay una sola línea roja en el log. El cliente cambió de bin porque
cambió la POBLACIÓN, no porque cambiara él.

**Bug B — el ruidoso.** Los cortes cambian y las etiquetas ya no calzan con el
mapa. El WoE cae a neutro. Hace mucho más daño… y se delata solo.

In [16]:
def etiquetas_ordenadas(spec):
    """Etiquetas de los bins en orden creciente, generadas con el MISMO `pd.cut`.

    pandas formatea los bordes con precision=3: reconstruirlas con el float
    completo daría cadenas distintas y el mapa de WoE no calzaría.
    """
    interiores = [float(c) for c in spec["cortes"][1:-1]]
    cortes = np.array([-np.inf, *interiores, np.inf])
    return [str(c) for c in pd.cut(pd.Series([np.nan]), cortes).cat.categories]

def puntuar_bins_del_lote(df, art):
    """BUG A: re-ajusta los cortes sobre el lote y mapea el WoE por posición."""
    lp = np.full(len(df), art["coeficientes"]["const"], dtype=float)
    sin_mapa = 0
    for v in art["variables"]:
        spec_dev, woe_dev = art["binning"][v], art["woe"][v]
        spec_lote = ajustar_bins(df[v])
        et = aplicar_bins(df[v], spec_lote)
        if spec_dev["tipo"] == "numerico" and spec_lote["tipo"] == "numerico":
            ord_dev = [e for e in etiquetas_ordenadas(spec_dev) if e in woe_dev]
            mapa = {b: woe_dev[ord_dev[min(k, len(ord_dev) - 1)]]
                    for k, b in enumerate(etiquetas_ordenadas(spec_lote))}
            if spec_dev.get("moda_aparte") and spec_lote.get("moda_aparte"):
                mapa[spec_lote["etiqueta_moda"]] = woe_dev.get(spec_dev["etiqueta_moda"], 0.0)
            if "MISSING" in woe_dev:
                mapa["MISSING"] = woe_dev["MISSING"]
        else:
            mapa = woe_dev
        serie = et.map(mapa).astype(float)
        sin_mapa += int(serie.isna().sum())
        lp += art["coeficientes"][v] * serie.fillna(art["woe_faltante"]).values
    return _cerrar_score(lp, art, sin_mapa, df.index)

def puntuar_woe_desmapeado(df, art):
    """BUG B: los cortes cambian y las etiquetas ya no calzan con el mapa."""
    lp = np.full(len(df), art["coeficientes"]["const"], dtype=float)
    sin_mapa = 0
    for v in art["variables"]:
        et = aplicar_bins(df[v], ajustar_bins(df[v]))
        serie = et.map(art["woe"][v]).astype(float)
        sin_mapa += int(serie.isna().sum())
        lp += art["coeficientes"][v] * serie.fillna(art["woe_faltante"]).values
    return _cerrar_score(lp, art, sin_mapa, df.index)

def _cerrar_score(lp, art, sin_mapa, idx):
    p = expit(logit(np.clip(expit(lp), 1e-12, 1 - 1e-12)) + art["calibracion"]["delta"])
    sc = score_desde_art(p, art)
    s = pd.DataFrame({"pd_calibrada": p, "score": sc}, index=idx)
    s.attrs["n_sin_mapa"] = sin_mapa
    return s

bug_a = puntuar_bins_del_lote(ttd, artefacto)
bug_b = puntuar_woe_desmapeado(ttd, artefacto)
n_woe = len(ttd) * len(artefacto["variables"])
correcto = salida_ttd["score"].values

filas = []
for nombre, s in [("A · cortes del lote, WoE por posición", bug_a),
                  ("B · etiquetas que ya no calzan", bug_b)]:
    d = s["score"].values - correcto
    cambia = (s["score"].values >= CUTOFF) != (correcto >= CUTOFF)
    filas.append({"bug": nombre, "delta_medio_pts": d.mean(),
                  "max_abs_delta": np.abs(d).max(),
                  "cambian_decision": int(cambia.sum()),
                  "pct_cambian": cambia.mean(),
                  "pct_woe_sin_mapa": s.attrs["n_sin_mapa"] / n_woe})
comparacion = pd.DataFrame(filas).set_index("bug")
print(comparacion.round(4).to_string())

                                       delta_medio_pts  max_abs_delta  cambian_decision  pct_cambian  pct_woe_sin_mapa
bug                                                                                                                   
A · cortes del lote, WoE por posición           4.3306        73.1585               587       0.0684            0.0000
B · etiquetas que ya no calzan                -25.1554       163.6877              2178       0.2537            0.8929


### La moraleja, que es contraintuitiva

El bug B hace **cuatro veces más daño** (25,4% de las decisiones contra 6,8%) —
y es el que se arregla el primer día, porque el 89% de los WoE cae sin mapa y
cualquier instrumentación mínima lo grita.

El bug A cambia el 6,8% de las decisiones **con cero alarmas**. Cuesta plata
todos los días y no aparece en ningún log.

> **El bug que se ve se arregla. El que no se ve se paga.** Por eso el binning
> se congela en el artefacto: no es prolijidad, es la diferencia entre un
> modelo y un generador de números.

In [17]:
# ¿quiénes son los 587 del bug A? Los que estaban cerca del cutoff.
d_a = bug_a["score"].values - correcto
cambian_a = (bug_a["score"].values >= CUTOFF) != (correcto >= CUTOFF)
print(f"distancia al cutoff de los que cambian: "
      f"media {np.abs(correcto[cambian_a] - CUTOFF).mean():.1f} pts")
print(f"distancia al cutoff del resto:          "
      f"media {np.abs(correcto[~cambian_a] - CUTOFF).mean():.1f} pts")
print("\nNo es aleatorio: el bug se come exactamente la franja donde la "
      "decisión estaba ajustada,\nque es donde el modelo aporta más valor.")

distancia al cutoff de los que cambian: media 10.0 pts
distancia al cutoff del resto:          media 60.7 pts

No es aleatorio: el bug se come exactamente la franja donde la decisión estaba ajustada,
que es donde el modelo aporta más valor.


---
## 6. Contrato de datos: qué bloquea y qué solo avisa

El modelo no sabe distinguir un dato bueno de uno malo: puntúa lo que le den.
El contrato es lo único entre su cartera y una corrida con basura adentro.

Dos severidades, y **la fija la magnitud, no la regla**:

- 🔴 **bloquea**: la corrida se aborta y el aborto queda registrado con su motivo.
- 🟡 **avisa**: se puntúa y queda constancia en el trail y en el tablero del mes.

Un 3% de valores fuera del rango de DEV es deriva de población. Un 100% es que
alguien cambió la unidad.

In [18]:
TOLERANCIA_FUERA_RANGO = 0.01   # 1% fuera de rango: avisa
FUERA_RANGO_BLOQUEANTE = 0.10   # 10%: bloquea (eso es un cambio de unidad)
MISSING_BLOQUEANTE = 0.25       # 1 de cada 4 casos sin dato: el lote no sirve
FACTOR_BLOQUEO = 4.0

def validar_contrato(df, contrato):
    hallazgos = []
    def anotar(v, sev, regla, detalle, valor, n_obs=None):
        hallazgos.append({"variable": v, "severidad": sev, "regla": regla,
                          "detalle": detalle, "valor": valor, "n_observado": n_obs})
    for v, c in contrato.items():
        if v not in df.columns:
            anotar(v, "🔴", "columna_faltante", "la columna no viene en el lote", None)
            continue
        s = df[v]
        num = es_numerica(s)
        if num != (c["tipo"] == "numerico"):
            anotar(v, "🔴", "tipo_incompatible",
                   f"esperado {c['tipo']}, llegó {'numerico' if num else 'categorico'}",
                   str(s.dtype))
            continue
        pm = float(s.isna().mean())
        if pm > c["pct_missing_max"]:
            # política publicada: bloquea al cruzar 4× SU tope O el 25% absoluto (OR)
            grave = pm > min(c["pct_missing_max"] * FACTOR_BLOQUEO, MISSING_BLOQUEANTE)
            anotar(v, "🔴" if grave else "🟡", "missing_excesivo",
                   f"{pm:.1%} > tope {c['pct_missing_max']:.1%}", pm, int(s.notna().sum()))
        obs = s.dropna()
        if c["tipo"] == "numerico" and len(obs):
            # sobre los OBSERVADOS: contar los NaN como "dentro" diluía el indicador
            fuera = float(((obs < c["minimo"]) | (obs > c["maximo"])).mean())
            if fuera > TOLERANCIA_FUERA_RANGO:
                anotar(v, "🔴" if fuera > FUERA_RANGO_BLOQUEANTE else "🟡",
                       "fuera_de_rango",
                       f"{fuera:.1%} de los observados fuera de "
                       f"[{c['minimo']:.4g}, {c['maximo']:.4g}]", fuera, int(len(obs)))
        elif c["tipo"] == "categorico":
            nuevas = sorted(set(obs.astype(str)) - set(c["categorias"]))
            if nuevas:
                pct = float(obs.astype(str).isin(nuevas).mean())
                anotar(v, "🔴" if pct > FUERA_RANGO_BLOQUEANTE else "🟡",
                       "categoria_nueva", f"{pct:.1%} en categorías nuevas: {nuevas[:4]}",
                       pct, int(len(obs)))
    return pd.DataFrame(hallazgos,
                        columns=["variable", "severidad", "regla", "detalle",
                                 "valor", "n_observado"])

hallazgos_ttd = validar_contrato(ttd, artefacto["contrato_datos"])
print("Bandeja REAL de hoy:")
print(hallazgos_ttd.to_string(index=False) if len(hallazgos_ttd) else "  (sin hallazgos)")

Bandeja REAL de hoy:
        variable severidad          regla                                  detalle    valor  n_observado
antiguedad_meses         🟡 fuera_de_rango 3.1% de los observados fuera de [6, 344] 0.031334         8585


### El contrato encontró algo real

No es un ejercicio: la bandeja de hoy trae un **3,1% de casos con
`antiguedad_meses` fuera del rango que el modelo vio en desarrollo**.

Y esto es lo importante: es **pariente** de las señales de la clase 5 — allá el
canario fue `deuda_interna_max_3m` (CSI 0,138); el CSI de `antiguedad_meses` es
bajo (0,002) y aun así el contrato la atrapa: **miran cosas distintas** (mezcla
de la distribución vs valores fuera del rango de desarrollo), por eso se tienen
ambos — y como la cohorte swap-in, disponible **hoy**, sin esperar los 12
meses del target. El contrato es el sistema de alerta más temprano que existe.

In [19]:
# El lote roto de laboratorio: los tres desastres clásicos de un feed
lote_roto = ttd.copy()
v0, v1, v2 = artefacto["variables"][:3]
lote_roto[v0] = lote_roto[v0] * 1000                                   # cambió la unidad
lote_roto.loc[lote_roto.index[:len(lote_roto) // 2], v1] = np.nan      # feed a la mitad
lote_roto = lote_roto.drop(columns=[v2])                               # columna que no llegó

h_roto = validar_contrato(lote_roto, artefacto["contrato_datos"])
print(h_roto.to_string(index=False))
print(f"\nbloqueantes: {(h_roto['severidad'] == '🔴').sum()} · "
      f"avisos: {(h_roto['severidad'] == '🟡').sum()}")

            variable severidad            regla                                             detalle    valor  n_observado
  uso_linea_prom_12m         🔴   fuera_de_rango 100.0% de los observados fuera de [0.07098, 0.5883] 1.000000       8585.0
meses_desde_mora_12m         🔴 missing_excesivo                                   50.0% > tope 1.0% 0.499942       4293.0
     uso_tc_prom_12m         🔴 columna_faltante                      la columna no viene en el lote      NaN          NaN
    antiguedad_meses         🟡   fuera_de_rango            3.1% de los observados fuera de [6, 344] 0.031334       8585.0

bloqueantes: 3 · avisos: 1


> ⚠️ Fíjense en qué **no** pasa acá: ninguno de los tres desastres lanza una
> excepción de Python. `uso_linea_prom_12m × 1000` es aritmética perfectamente
> válida. El feed a la mitad son NaN, que pandas maneja sin quejarse. Sin
> contrato, el lote se puntúa entero y el reporte del mes sale con números
> inventados.

---
## 7. La corrida de producción, sellada de crudo a score

Un audit trail responde tres preguntas: **qué se ejecutó, sobre qué datos, y en
qué orden**. Cada evento sella al anterior con su hash. Honestidad de alcance:
la matriz se **fabricó arriba** (la receta de las clases 1–2, versionada); la
corrida registra las crudas, la construcción y el lote **con sus hashes** — la
versión donde la fabricación misma vive dentro del job es la corrida end-to-end
de la pista industrial (sección 10).

In [20]:
class RegistroAuditoria:
    """Trail con hashes encadenados + verificación estricta."""

    def __init__(self, corrida_id, semilla_ts=0):
        self.corrida_id = corrida_id
        self.eventos = []
        self._ts = semilla_ts       # reloj lógico: determinista

    def registrar(self, paso, evento, **payload):
        previo = self.eventos[-1]["hash"] if self.eventos else "0" * 64
        cuerpo = {"n": len(self.eventos) + 1, "corrida_id": self.corrida_id,
                  "t": self._ts, "paso": paso, "evento": evento,
                  "payload": payload, "hash_previo": previo}
        cuerpo["hash"] = hash_json(cuerpo)
        self.eventos.append(cuerpo)
        self._ts += 1
        return cuerpo

    def sello(self):
        """(hash terminal, nº de eventos): el ancla que se archiva FUERA del log."""
        if not self.eventos:
            return None, 0
        return self.eventos[-1]["hash"], len(self.eventos)

    def verificar_cadena(self, sello=None,
                         cierres=("corrida_terminada", "corrida_abortada")):
        if not self.eventos:
            return False, "trail vacío: una corrida sin eventos no es evidencia"
        previo = "0" * 64
        for k, e in enumerate(self.eventos, start=1):
            if e["n"] != k:
                return False, f"numeración rota en la posición {k}"
            if e["corrida_id"] != self.corrida_id:
                return False, f"el evento {k} pertenece a otra corrida"
            cuerpo = {c: v for c, v in e.items() if c != "hash"}
            if cuerpo["hash_previo"] != previo:
                return False, f"el evento {k} no encadena con el anterior"
            if hash_json(cuerpo) != e["hash"]:
                return False, f"el contenido del evento {k} fue alterado"
            previo = e["hash"]
        if self.eventos[-1]["evento"] not in cierres:
            return False, "el trail no termina en un evento de cierre (fue truncado)"
        if sello is not None:
            h, n = sello
            if len(self.eventos) != n:
                return False, f"el sello externo declara {n} eventos, el trail trae {len(self.eventos)}"
            if self.eventos[-1]["hash"] != h:
                return False, "el hash terminal no coincide con el sello externo"
        return True, None

    def tabla(self):
        return pd.DataFrame([{"n": e["n"], "paso": e["paso"], "evento": e["evento"],
                              "hash": e["hash"][:12]} for e in self.eventos])

def snapshot_entorno():
    import scipy, sklearn, statsmodels
    return {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "statsmodels": statsmodels.__version__}

print("clase RegistroAuditoria lista")

clase RegistroAuditoria lista


In [21]:
def hash_tabla_cruda(df):
    return hash_df(df)

def corrida_produccion(art, lote, tablas_crudas, registro=None):
    """La corrida completa, con cada eslabón sellado."""
    reg = registro or RegistroAuditoria(f"{art['modelo_id']}-{art['fecha_construccion']}")
    reg.registrar("inicio", "corrida_iniciada", modelo=art["modelo_id"],
                  version=art["version"], entorno=snapshot_entorno())
    reg.registrar("datos", "fuentes_crudas_leidas",
                  tablas={n: hash_tabla_cruda(t) for n, t in tablas_crudas.items()})
    # sobre las columnas PRESENTES: si falta una, queremos el mensaje del
    # contrato, no un KeyError críptico tres líneas antes
    presentes = [c for c in art["variables"] if c in lote.columns]
    # la matriz se fabricó ARRIBA con la receta versionada de las clases 1-2;
    # la corrida la sella como eslabón propio: receta + hash de lo construido
    reg.registrar("datos", "matriz_construida",
                  receta="ventanas ancladas a t0-1 + fotos + ratios (receta C1-C2)",
                  n=int(len(lote)), n_variables=len(presentes),
                  hash_matriz=hash_df(lote[presentes]))
    reg.registrar("datos", "lote_seleccionado", muestra="TTD", n=int(len(lote)),
                  n_variables_presentes=len(presentes),
                  hash_lote=hash_df(lote[presentes]))
    h = validar_contrato(lote, art["contrato_datos"])
    bloqueantes = int((h["severidad"] == "🔴").sum())
    reg.registrar("contrato", "contrato_validado", n_hallazgos=int(len(h)),
                  bloqueantes=bloqueantes,
                  hallazgos=h.to_dict("records"))   # detalle completo: qué, cuánto, severidad
    if bloqueantes:
        reg.registrar("contrato", "corrida_abortada", motivo="hallazgos bloqueantes",
                      detalle=h[h["severidad"] == "🔴"][["variable", "regla"]]
                      .to_dict("records"))
        raise ValueError(f"contrato incumplido: {bloqueantes} hallazgo(s) 🔴")
    reg.registrar("modelo", "artefacto_cargado", hash_artefacto=hash_json(art),
                  n_variables=len(art["variables"]), delta=art["calibracion"]["delta"])
    s = puntuar(lote, art)
    reg.registrar("scoring", "lote_puntuado", n=int(len(s)),
                  score_medio=float(s["score"].mean()),
                  pd_media=float(s["pd_calibrada"].mean()),
                  tasa_aprobacion=float((s["decision"] == "aprobar").mean()),
                  n_a_revision=int((s["decision"] == "revisar").sum()),
                  hash_salida=hash_df(s.astype({"banda": "string"})))
    reg.registrar("fin", "corrida_terminada", estado="ok")
    return s, reg

CRUDAS = {"clientes": clientes, "solicitudes": solicitudes,
          "comportamiento": comportamiento, "bureau": bureau}
salida, registro = corrida_produccion(artefacto, ttd, CRUDAS)
SELLO = registro.sello()

print(registro.tabla().to_string(index=False))
print(f"\nsello de la corrida: {SELLO[0][:16]}… · {SELLO[1]} eventos")
print(f"aprobación: {(salida['decision'] == 'aprobar').mean():.1%} · "
      f"a revisión: {(salida['decision'] == 'revisar').sum()} casos · "
      f"PD media calibrada: {salida['pd_calibrada'].mean():.2%}")

 n     paso                evento         hash
 1   inicio      corrida_iniciada d69d374e32d1
 2    datos fuentes_crudas_leidas 304285ef0b22
 3    datos     matriz_construida f510389f3e23
 4    datos     lote_seleccionado 4da7a29add7e
 5 contrato     contrato_validado 8cf2b79af967
 6   modelo     artefacto_cargado f0b3306f8c59
 7  scoring         lote_puntuado 04354706bb17
 8      fin     corrida_terminada b92e00030155

sello de la corrida: b92e00030155f5bd… · 8 eventos
aprobación: 74.6% · a revisión: 0 casos · PD media calibrada: 6.31%


In [22]:
# El contrato ABORTA — no puntúa igual "porque total son pocos casos"
try:
    corrida_produccion(artefacto, lote_roto, CRUDAS,
                       RegistroAuditoria("intento-lote-roto"))
except ValueError as e:
    print(f"❌ {e}")
    print("\nY el aborto queda registrado, con su motivo: una corrida que no "
          "ocurrió\ntambién es información de auditoría.")

❌ contrato incumplido: 3 hallazgo(s) 🔴

Y el aborto queda registrado, con su motivo: una corrida que no ocurrió
también es información de auditoría.


---
## 8. Romper el trail de dos maneras (y por qué existe el sello)

Acá es donde la mayoría de las presentaciones sobre "trazabilidad" miente por
omisión. Vamos a ser exactos sobre qué detecta esta cadena y qué no.

In [23]:
import copy

# --- ATAQUE 1: el "arreglito" de última hora ---------------------------
# Alguien no quiere explicar una aprobación del 74,6% y la sube al 99%.
torpe = copy.deepcopy(registro)
n_objetivo = next(e["n"] for e in torpe.eventos if e["evento"] == "lote_puntuado")
torpe.eventos[n_objetivo - 1]["payload"]["tasa_aprobacion"] = 0.99

ok, motivo = torpe.verificar_cadena()
print(f"ataque 1 (editar sin resellar) → cadena íntegra: {ok}")
print(f"  motivo: {motivo}")

ataque 1 (editar sin resellar) → cadena íntegra: False
  motivo: el contenido del evento 7 fue alterado


In [24]:
# --- ATAQUE 2: el prolijo ---------------------------------------------
# El mismo cambio, pero recalculando TODA la cola de hashes.
prolijo = copy.deepcopy(registro)
prolijo.eventos[n_objetivo - 1]["payload"]["tasa_aprobacion"] = 0.99
previo = "0" * 64
for e in prolijo.eventos:
    e["hash_previo"] = previo
    e["hash"] = hash_json({k: v for k, v in e.items() if k != "hash"})
    previo = e["hash"]

ok_sin_sello, _ = prolijo.verificar_cadena()
ok_con_sello, motivo_sello = prolijo.verificar_cadena(sello=SELLO)
print(f"ataque 2 (editar Y resellar) → cadena sola dice íntegra: {ok_sin_sello}")
print(f"                             → contra el sello externo: {ok_con_sello}")
print(f"  motivo: {motivo_sello}")

ataque 2 (editar Y resellar) → cadena sola dice íntegra: True
                             → contra el sello externo: False
  motivo: el hash terminal no coincide con el sello externo


### Lo que esto significa, dicho sin adornos

- La cadena de hashes detecta la **edición sin resellado**: el arreglito de
  última hora, que es el caso real y frecuente. Eso ya justifica montarla.
- **NO detecta** a quien pueda reescribir el archivo completo. Sin clave
  secreta ni ancla externa, esto es detección de manipulación descuidada, no
  prueba criptográfica.
- La pieza que lo convierte en evidencia es el **sello**: hash del último
  evento + número de eventos, archivado **fuera** del log — en el expediente,
  en almacenamiento append-only, o firmado por un tercero.

> Un control que no sabe qué NO cubre es peor que no tenerlo: da confianza sin
> dar garantía. Si mañana les venden una herramienta de "audit trail
> inmutable", la pregunta es: ¿dónde está el ancla?

In [25]:
# La verificación también caza el borrado y el truncamiento
truncado = copy.deepcopy(registro)
truncado.eventos = truncado.eventos[:-1]        # se pierde el cierre
print("truncado  →", truncado.verificar_cadena()[1])

vacio = RegistroAuditoria("sin-eventos")
print("vacío     →", vacio.verificar_cadena()[1])

truncado  → el trail no termina en un evento de cierre (fue truncado)
vacío     → trail vacío: una corrida sin eventos no es evidencia


---
## 9. Lineage: los cuatro eslabones con su hash

*Lineage* es poder responder **"de qué datos y con qué modelo salió este
número"** sin depender de la memoria de nadie.

In [26]:
eslabones = []
for e in registro.eventos:
    if e["evento"] == "fuentes_crudas_leidas":
        for t, h in e["payload"]["tablas"].items():
            eslabones.append({"eslabón": f"crudo · {t}", "hash": h[:16] + "…"})
    for k, v in e["payload"].items():
        if k.startswith("hash"):
            eslabones.append({"eslabón": k.replace("hash_", ""), "hash": v[:16] + "…"})
print(pd.DataFrame(eslabones).to_string(index=False))

               eslabón              hash
      crudo · clientes 2e1869be4385b69f…
   crudo · solicitudes 1f0824a00bc07b1e…
crudo · comportamiento b6f4dec64ae12be7…
        crudo · bureau 03266db829ab38aa…
                matriz 7084c06d91c126e7…
                  lote 7084c06d91c126e7…
             artefacto 98845d693591d141…
                salida 2b36aba7faa78336…


In [27]:
# Determinismo: repetir la corrida da los MISMOS hashes, no "parecidos"
_, registro_2 = corrida_produccion(artefacto, ttd, CRUDAS,
                                   RegistroAuditoria(registro.corrida_id))
h1 = [e["hash"] for e in registro.eventos if e["evento"] != "corrida_iniciada"]
h2 = [e["hash"] for e in registro_2.eventos if e["evento"] != "corrida_iniciada"]
print(f"hashes idénticos entre dos corridas: {h1 == h2}")
print("\n(el primer evento incluye el snapshot de entorno, que puede variar "
      "entre máquinas:\npor eso se compara desde el segundo — y por eso el "
      "entorno va DECLARADO en la ficha)")

hashes idénticos entre dos corridas: True

(el primer evento incluye el snapshot de entorno, que puede variar entre máquinas:
por eso se compara desde el segundo — y por eso el entorno va DECLARADO en la ficha)


---
## 10. El model card, generado desde la corrida

Una página que un gerente lee en tres minutos y que un validador usa como
índice. Responde tres preguntas:

1. ¿Qué hace y para quién?
2. ¿Qué tan bien lo hace, y sobre qué evidencia?
3. ¿Qué **no** se puede concluir de él?

La regla que la hace útil: **se genera DESDE la corrida**, no se escribe aparte
de memoria. Un informe que no corresponde a los números del notebook es
invalidante — y es el error más común.

In [28]:
card = {
    "identidad": {
        "modelo": artefacto["modelo_id"], "version": artefacto["version"],
        "tipo": "Scorecard logístico de admisión (PD a 12 meses)",
        "cartera": artefacto["cartera"],
        "fecha_construccion": artefacto["fecha_construccion"],
        "responsable_desarrollo": "Equipo de Modelos de Riesgo",
        "hash_artefacto": HASH_ARTEFACTO,
        "corrida_id": registro.corrida_id,
        "sello_trail": f"{SELLO[0]} ({SELLO[1]} eventos)"},
    "proposito": {
        "uso_previsto": f"Decisión de admisión de crédito de consumo con cutoff "
                        f"{artefacto['politica']['cutoff']} sobre el score.",
        "usuarios": "Motor de originación (automático) y mesa de excepciones (manual).",
        "usos_no_previstos": [
            "Provisiones IFRS 9 (requiere PD lifetime, no a 12 meses).",
            "Cobranza o gestión de cartera vigente (es un modelo de admisión).",
            "Precios diferenciados por riesgo (no fue validado para ese uso).",
            "Segmentos fuera de consumo (hipotecario, comercial, microempresa)."]},
    "datos": {
        "poblacion": "Solicitudes de consumo aprobadas, 2024-07 a 2025-06.",
        "definicion_target": "Default = 90+ DPD dentro de 12 meses del punto de observación.",
        "muestras": {m: int(len(MUESTRAS[m])) for m in ("DEV", "HO", "OOT", "TTD")},
        "tendencia_central": float(TC),
        "variables_finales": artefacto["variables"]},
    "desempeno": {
        "gini": {m: float(metricas.loc[m, "gini"]) for m in ("DEV", "HO", "OOT")},
        "ks": {m: float(metricas.loc[m, "ks"]) for m in ("DEV", "HO", "OOT")},
        "psi_score_ttd": float(psi_ttd),
        "backtesting_global_oot_p": float(binom_global.pvalue),
        "tablero_clase_5": "4 🟢 · 5 🟡 · 0 🔴"},
    "limitaciones": [
        "Modelado sobre solicitudes APROBADAS: no observa el desempeño de las "
        "rechazadas (33,1% de las solicitudes históricas).",
        "La tendencia central se ancló con las 12 cosechas maduras, incluidas las de "
        "OOT: la mejora del nivel en OOT no es validación independiente.",
        "Hosmer-Lemeshow en OOT queda amarillo (p simulado 0,012): la FORMA de la "
        "calibración se torció en los tramos de PD baja.",
        "Cohorte swap-in (n=128): mora 9,4% contra PD 5,1% (p 0,04) — subestima en el "
        "territorio que la política vigente rechazaba.",
        "El desempeño en TTD no está verificado: esa ventana madura en ago-2027.",
        "El audit trail encadenado detecta edición sin resellado, no la reescritura "
        "completa del log: la evidencia exige el sello externo archivado aparte."],
    "gobierno": {
        "estado": "en validación independiente",
        "proxima_revalidacion": "2027-09-10",
        "frecuencia_monitoreo": "mensual (discriminación y estabilidad) · "
                                "trimestral (calibración)",
        "tablero_vive_en": "el expediente del modelo"},
    "entorno": snapshot_entorno(),
}

def card_markdown(card):
    L = [f"# Model Card — {card['identidad']['modelo']} "
         f"v{card['identidad']['version']}", ""]
    titulos = {"identidad": "1. Identidad", "proposito": "2. Propósito y uso previsto",
               "datos": "3. Datos", "desempeno": "4. Desempeño",
               "limitaciones": "5. Limitaciones", "gobierno": "6. Gobierno",
               "entorno": "7. Entorno de construcción"}
    for k, t in titulos.items():
        L += [f"## {t}", ""]
        v = card[k]
        if isinstance(v, list):
            L += [f"- {x}" for x in v]
        else:
            for k2, v2 in v.items():
                etiqueta = k2.replace("_", " ").capitalize()
                if isinstance(v2, (dict, list)):
                    L.append(f"- **{etiqueta}**:")
                    items = v2.items() if isinstance(v2, dict) else enumerate(v2)
                    L += [f"  - {a}: `{b}`" if isinstance(v2, dict) else f"  - {b}"
                          for a, b in items]
                else:
                    L.append(f"- **{etiqueta}**: {v2}")
        L.append("")
    return "\n".join(L)

texto_card = card_markdown(card)
print(texto_card[:1500])
print(f"\n… ({len(texto_card.splitlines())} líneas en total)")

# Model Card — austral-scorecard-consumo v1.0.0

## 1. Identidad

- **Modelo**: austral-scorecard-consumo
- **Version**: 1.0.0
- **Tipo**: Scorecard logístico de admisión (PD a 12 meses)
- **Cartera**: Crédito de consumo — Banco Austral
- **Fecha construccion**: 2026-09-10
- **Responsable desarrollo**: Equipo de Modelos de Riesgo
- **Hash artefacto**: 98845d693591d141112674264774a76b7088d1b8d73fb7cc626cbb71f8171094
- **Corrida id**: austral-scorecard-consumo-2026-09-10
- **Sello trail**: b92e00030155f5bd2b653c427efa6d0e7defb852aee0eceaa2dee21ba5b96415 (8 eventos)

## 2. Propósito y uso previsto

- **Uso previsto**: Decisión de admisión de crédito de consumo con cutoff 560 sobre el score.
- **Usuarios**: Motor de originación (automático) y mesa de excepciones (manual).
- **Usos no previstos**:
  - Provisiones IFRS 9 (requiere PD lifetime, no a 12 meses).
  - Cobranza o gestión de cartera vigente (es un modelo de admisión).
  - Precios diferenciados por riesgo (no fue validado para ese u

In [29]:
# se archiva junto al artefacto y al sello: eso es el expediente
with open("model_card_austral.md", "w", encoding="utf-8") as f:
    f.write(texto_card)
with open("modelo_austral.json", "w", encoding="utf-8") as f:
    json.dump(artefacto, f, indent=1, ensure_ascii=False)
with open("sello_austral.json", "w", encoding="utf-8") as f:
    json.dump({"hash_terminal": SELLO[0], "n_eventos": SELLO[1],
               "corrida_id": registro.corrida_id}, f, indent=1)
with open("audit_trail_austral.jsonl", "w", encoding="utf-8") as f:
    for e in registro.eventos:
        f.write(json.dumps(e, ensure_ascii=False, sort_keys=True, default=str) + "\n")
print("expediente escrito: model_card_austral.md · modelo_austral.json · "
      "audit_trail_austral.jsonl · sello_austral.json")
print("(el sello viaja aquí como archivo adyacente — didáctico; en producción "
      "va a un medio que\neste equipo no pueda reescribir: append-only, custodio, firma)")

expediente escrito: model_card_austral.md · modelo_austral.json · audit_trail_austral.jsonl · sello_austral.json
(el sello viaja aquí como archivo adyacente — didáctico; en producción va a un medio que
este equipo no pueda reescribir: append-only, custodio, firma)


---
## 11. El resumen ejecutivo de una página

El informe técnico es para el validador. El model card, para quien decide si
usar el modelo. El resumen ejecutivo es para **el comité**, y responde una sola
pregunta: *¿qué firmo y qué me pasa si lo firmo?*

Cinco bloques, en este orden. El comité lee de arriba hacia abajo y se detiene
cuando ya sabe qué votar.

In [30]:
aprob = (salida["decision"] == "aprobar").mean()
resumen = f"""
RESUMEN EJECUTIVO — {artefacto['modelo_id']} v{artefacto['version']}
{'=' * 72}

1. QUÉ SE PIDE APROBAR
   Uso del scorecard de admisión v{artefacto['version']} con cutoff
   {artefacto['politica']['cutoff']} para crédito de consumo, por 12 meses.

2. QUÉ GANA EL BANCO
   Sobre la ventana OOT y a igual aprobación que la política de knock-outs de
   referencia (90,2% de las cursadas), la mora de la cartera originada baja
   ~22% (4,48% → 3,48%). Con el cutoff 560 propuesto, la bandeja actual (TTD)
   se aprueba al {aprob:.1%}.
   Discriminación: Gini {metricas.loc['OOT', 'gini']:.3f} en la muestra fuera de
   tiempo, sin caída concluyente al 95%.

3. QUÉ SE SABE QUE NO FUNCIONA
   - Subestima el riesgo de solicitantes que la política actual rechazaba
     (mora observada 9,4% contra 5,1% previsto).
   - No observa a los rechazados históricos (33,1% de las solicitudes).
   - El desempeño de la bandeja de hoy no se conocerá hasta ago-2027.

4. QUÉ SE VA A VIGILAR Y CON QUÉ GATILLO
   Tablero mensual (discriminación y estabilidad) y trimestral (calibración).
   Si el backtesting queda amarillo dos trimestres seguidos, se recalibra el δ
   con acta. Si la caída de Gini supera 30%, o el KS baja de 0,20, o el PSI del
   score supera 0,25, se abre re-desarrollo.

5. QUÉ SE PIDE DE VUELTA
   Confirmación del apetito de riesgo para el tramo D-E y designación del
   responsable de la revalidación de 2027.
"""
print(resumen)


RESUMEN EJECUTIVO — austral-scorecard-consumo v1.0.0

1. QUÉ SE PIDE APROBAR
   Uso del scorecard de admisión v1.0.0 con cutoff
   560 para crédito de consumo, por 12 meses.

2. QUÉ GANA EL BANCO
   Sobre la ventana OOT y a igual aprobación que la política de knock-outs de
   referencia (90,2% de las cursadas), la mora de la cartera originada baja
   ~22% (4,48% → 3,48%). Con el cutoff 560 propuesto, la bandeja actual (TTD)
   se aprueba al 74.6%.
   Discriminación: Gini 0.675 en la muestra fuera de
   tiempo, sin caída concluyente al 95%.

3. QUÉ SE SABE QUE NO FUNCIONA
   - Subestima el riesgo de solicitantes que la política actual rechazaba
     (mora observada 9,4% contra 5,1% previsto).
   - No observa a los rechazados históricos (33,1% de las solicitudes).
   - El desempeño de la bandeja de hoy no se conocerá hasta ago-2027.

4. QUÉ SE VA A VIGILAR Y CON QUÉ GATILLO
   Tablero mensual (discriminación y estabilidad) y trimestral (calibración).
   Si el backtesting queda amarillo 

> 🎯 **Prueba de fuego**: si alguien lo lee y no sabe qué tiene que decidir,
> no es un resumen ejecutivo — es un abstract. Y sin números concretos no es
> ejecutivo: es una carta de intenciones.

---
## 12. El atajo: `governance`, `audit` y `report` de `nikodym`

Todo lo de hoy lo escribimos a mano — y hay que saber escribirlo, porque es lo
que un validador va a revisar línea por línea. Ahora el atajo.

El estudio F1 que corrimos en las clases 3, 4 y 5 es el mismo. Lo único que
cambia es que hoy **encendemos dos secciones del config** que hasta ahora
venían apagadas: `audit` y `governance`.

In [31]:
try:
    import nikodym
except ImportError:
    !pip install -q "nikodym[scoring]==1.11.0"
    import nikodym

print("nikodym", nikodym.__version__)

nikodym 1.11.0


In [32]:
# ---- la receta de siempre: el pool estable + el config del preset F1 ----
def psi_simple(esperado, actual, bins=10):
    esperado, actual = pd.Series(esperado), pd.Series(actual)
    if esperado.dtype == object:
        e = esperado.fillna("MISSING").astype(str)
        a = actual.fillna("MISSING").astype(str)
    else:
        cortes = np.unique(np.nanquantile(esperado.dropna(), np.linspace(0, 1, bins + 1)))
        if len(cortes) < 3:
            return np.nan
        cortes[0], cortes[-1] = -np.inf, np.inf
        e = pd.cut(esperado, cortes).astype(str).where(esperado.notna(), "MISSING")
        a = pd.cut(actual, cortes).astype(str).where(actual.notna(), "MISSING")
    cats = sorted(set(e) | set(a))
    pe = e.value_counts(normalize=True).reindex(cats).fillna(0) + 1e-4
    pa = a.value_counts(normalize=True).reindex(cats).fillna(0) + 1e-4
    return float(((pa - pe) * np.log(pa / pe)).sum())

psi_ttd_vars = pd.Series({v: psi_simple(dev[v], ttd[v]) for v in predictoras})
pool = list(psi_ttd_vars[psi_ttd_vars <= 0.25].index)
categoricas = [c for c in pool if X[c].dtype == object]

entrenamiento = X[X["muestra"].isin(["DEV", "HO", "OOT"])
                  & X["malo"].notna() & ~X["indeterminado"]].copy()
entrenamiento["malo"] = entrenamiento["malo"].astype(int)
entrenamiento["cohorte"] = entrenamiento["fecha_solicitud"].astype(str)
frame = entrenamiento[["id_solicitud", "cohorte", "malo"] + pool].set_index("id_solicitud")
frame.to_parquet("matriz_austral_estables.parquet")
print(f"pool estable: {len(pool)} variables · frame {frame.shape}")

pool estable: 94 variables · frame (6723, 96)


In [33]:
from nikodym.core.config import NikodymConfig
from nikodym.ui.presets import standard_preset

cfg = standard_preset()["config"]                     # preset F1 (el de C3, C4 y C5)
cfg["data"]["load"]["source"] = "matriz_austral_estables.parquet"
cfg["data"]["schema"]["columns"] = (
    [{"name": "cohorte", "dtype": "str"}, {"name": "malo", "dtype": "int"}]
    + [{"name": c,
        "dtype": ("str" if c in categoricas
                  else "int" if pd.api.types.is_integer_dtype(frame[c]) else "float"),
        "nullable": True}
       for c in pool])
cfg["data"]["schema"]["index_col"] = "id_solicitud"
cfg["data"]["target"]["bad_rule"] = {"all_of": [{"col": "malo", "op": "==", "value": 1}]}
cfg["data"]["partition"]["strategy"] = {
    "type": "cohort", "cohort_col": "cohorte",
    "oot_cohorts": ["2025-03", "2025-04", "2025-05", "2025-06"],
    "holdout_fraction": 0.3}
cfg["binning"]["feature_columns"] = pool
cfg["binning"]["categorical_columns"] = categoricas
cfg["selection"]["min_iv"] = 0.10
cfg["selection"]["correlation"]["threshold"] = 0.70
cfg["calibration"]["target_pd"] = round(TC, 6)
cfg["calibration"]["anchor_source"] = "historical_default_rate"

# ---- LO NUEVO DE HOY: encender audit y governance ----
# El trail es ACUMULATIVO: si quedó uno de una corrida anterior, los eventos se
# suman y el conteo deja de corresponder a esta corrida. Se parte de cero.
from pathlib import Path
Path("audit_trail.jsonl").unlink(missing_ok=True)

cfg["audit"] = {"enabled": True, "trail_filename": "audit_trail.jsonl",
                "capture_environment": True}
cfg["governance"] = {
    "model_name": "austral-scorecard-consumo",
    "cartera": "Consumo Banco Austral",
    "motor": "scoring", "fase": "F1",
    "estado_validacion": "en_validacion",
    "author": "Equipo de Modelos — Banco Austral",
    "purpose": "Scorecard de admisión para crédito de consumo, cutoff 560.",
    "assumptions": ("Default = 90+ DPD en 12 meses.",
                    "Población: solicitudes aprobadas 2024-07 a 2025-06."),
    "limitations": ("Modelado sobre aprobados: no observa las rechazadas.",),
    "review_period_months": 12,
}

config = NikodymConfig.model_validate(cfg)
study = nikodym.run(config)                            # ~1-2 min
assert study.run_context.status == "done", study.run_context.error
print(f"estudio F1 completo ✅ · run_id {study.run_context.run_id}")

estudio F1 completo ✅ · run_id bcfebf0b337d4e9ab6af27a40355f00b


In [34]:
# ---- el audit trail que generó solo ----
from nikodym.audit import read_trail
import collections

eventos = read_trail("audit_trail.jsonl")
print(f"{len(eventos)} eventos registrados automáticamente")
print(collections.Counter(e.kind for e in eventos))
print("\nprimeras decisiones registradas:")
for e in [x for x in eventos if x.kind == "decision"][:4]:
    print(f"  {e.step or '—':12s} {e.payload['regla']:24s} → {e.payload['accion']}")

353 eventos registrados automáticamente
Counter({'decision': 296, 'artifact': 55, 'run_start': 1, 'run_end': 1})

primeras decisiones registradas:
  —            partition_strategy       → aplicar_estrategia
  —            partition_summary        → reportar_particiones
  —            monotonia_auto_resuelta  → registrar_tendencia
  —            monotonia_auto_resuelta  → registrar_tendencia


In [35]:
# ---- el lineage bundle: la identidad completa de la corrida ----
bundle = study.lineage_bundle()
for k, v in bundle.model_dump().items():
    if k != "library_versions":
        print(f"{k:22s} {str(v)[:70]}")

git_sha                89626dc28b37ce53cfb288715975e8044401f643
git_dirty              True
data_hash              352f9355ac182f4fd370fa331d3cf697da1d39c6ac762886798bc2a94541bcc6
config_hash            ae0fe0c8e166b4df681c991d9e212d45221c01026b1ae1ef16a37026d41c9590
root_seed              20240706
uv_lock_hash           None
determinism_caveats    ['working tree git sucio: cambios sin commitear no reconstruibles']
created_at             2026-08-29 00:05:56.276417+00:00
schema_version         1.0.0
injected_artifacts     ()


In [36]:
# ---- el model card, construido desde el estudio y su trail ----
from nikodym.governance import GovernanceConfig, ModelCardBuilder

gcfg = GovernanceConfig.model_validate(cfg["governance"])
card_nk = ModelCardBuilder(gcfg).build(study, trail_path="audit_trail.jsonl")

print(f"run_id:            {card_nk.run_id}")
print(f"config_hash:       {card_nk.config_hash[:24]}…")
print(f"data_hash:         {card_nk.data_hash[:24]}…")
print(f"root_seed:         {card_nk.root_seed}")
print(f"decisiones:        {len(card_nk.decisions)} registradas desde el trail")
print(f"revisión:          {card_nk.review_date:%Y-%m-%d} → "
      f"{card_nk.next_review_date:%Y-%m-%d}")
print("\nlimitaciones que quedaron en la ficha:")
for l in card_nk.limitations:
    print(f"  · {l}")

run_id:            bcfebf0b337d4e9ab6af27a40355f00b
config_hash:       ae0fe0c8e166b4df681c991d…
data_hash:         352f9355ac182f4fd370fa33…
root_seed:         20240706
decisiones:        296 registradas desde el trail
revisión:          2026-08-29 → 2027-08-29

limitaciones que quedaron en la ficha:
  · Modelado sobre aprobados: no observa las rechazadas.
  · working tree git sucio: cambios sin commitear no reconstruibles
  · lineage parcial: sin hash de uv.lock


### 🔍 El detalle que vale la clase entera

Miren las limitaciones de arriba. Nosotros declaramos **una** en el config
(«modelado sobre aprobados»). La ficha trae más — y no las escribimos nosotros.

Cuáles aparecen depende de **dónde se ejecute**:

- En **Colab** (donde corre esta demo): *«git no disponible: la corrida no tiene
  SHA de origen»*. No hay repositorio, así que no se puede probar qué versión
  del código produjo estos números.
- Corriendo **dentro de un repositorio con cambios sin commitear**: *«working
  tree git sucio: cambios sin commitear no reconstruibles»*. Hay SHA, pero el
  código que corrió no es exactamente el que ese SHA describe.
- En cualquiera de los dos: *«lineage parcial: sin hash de uv.lock»* — no hay
  archivo de bloqueo de dependencias, así que el entorno no es reconstruible
  al detalle.

En los tres casos la herramienta **detectó que no podía probar el origen del
código** y lo escribió en la ficha, sin que nadie se lo pidiera.

Eso es exactamente lo que debe hacer una ficha honesta: **declarar lo que no
puede demostrar**. Y es lo contrario de lo que hace la mayoría de los informes,
donde lo que falta simplemente no se menciona.

> Pregunta de comité, entonces: *«¿esta ficha declara sus propios límites, o
> solo los que alguien se acordó de escribir?»*

In [37]:
# ---- el informe: 7 secciones y 3 anexos, la estructura estándar ----
import re
from pathlib import Path

html = Path("reports/scorecard_report.html").read_text(encoding="utf-8")
secciones = [re.sub(r"<[^>]+>", "", s) for s in
             re.findall(r"<h2[^>]*>(.*?)</h2>", html)]
print(f"informe generado: {len(html):,} caracteres · {html.count('<table')} tablas")
print("\nsecciones:")
for s in secciones:
    print(f"  · {s}")

informe generado: 590,328 caracteres · 112 tablas

secciones:
  · Resumen ejecutivo
  · Índice
  · 1Introducción
  · 2Contexto del modelo y de la cartera
  · 3Metodología
  · 4Resultados
  · 5Validación formal
  · 6Conclusiones y recomendación
  · 7Limitaciones y supuestos
  · Anexo A —Lineage y reproducibilidad
  · Anexo B —Tablas detalladas
  · Anexo C —Parámetros completos


### Esa lista no es una casualidad

**Resumen ejecutivo · Introducción · Contexto del modelo y de la cartera ·
Metodología · Resultados · Validación formal · Conclusiones y recomendación ·
Limitaciones y supuestos**, más los anexos de **lineage**, **tablas detalladas**
y **parámetros completos**.

Es la misma estructura de la plantilla del informe técnico que vienen llenando
desde la clase 1 — porque **es el estándar de la industria**, no un formato del
curso. La plantilla está en `04_proyecto/`, y cada sección dice de qué clase
salió su contenido.

| | A mano (hoy) | `nikodym` |
|---|---|---|
| Artefacto | JSON explícito que escribimos nosotros | interno al estudio |
| Audit trail | 8 eventos, los que elegimos | 353 eventos, automáticos |
| Anclaje del trail | sello externo (hash + nº eventos) | `config_hash` + `data_hash` |
| Contrato de datos | reglas propias con dos severidades | `schema` de la config |
| Model card | 7 secciones escritas por nosotros | se arma desde el estudio y se auto-degrada |
| Informe | la plantilla del curso | HTML/qmd con 7 secciones + 3 anexos |

Ninguna de las dos columnas reemplaza a la otra. La izquierda es lo que hay que
**entender**; la derecha es lo que hay que **usar** cuando el modelo es uno de
treinta y la auditoría es en marzo.

---
## Cierre — y el curso completo

Hoy convertimos el notebook en cuatro cosas que sobreviven sin nosotros:

1. **Un artefacto** de 5.463 caracteres, en JSON estándar, con el binning
   congelado — y la prueba de paridad que demuestra que produce el mismo número
   que el notebook.
2. **Un contrato** que bloquea lo grave y avisa lo leve, y que ya encontró algo
   real en la bandeja de hoy (3,1% fuera del rango de desarrollo).
3. **Un expediente**: trail encadenado con su sello externo, lineage de cuatro
   eslabones, model card, informe técnico y resumen ejecutivo.
4. **Un gobierno**: quién decide qué, con qué gatillo y en qué plazo.

### Lo que se llevan del curso

Seis clases: diseñar, fabricar, elegir, calibrar, validar, gobernar. Lo que se
llevan no es un scorecard — es el **criterio para juzgar cualquiera**.

### Para el comité (los 3 mensajes de hoy)

1. **«Muéstrenme la corrida que produjo este número.»** Si la respuesta necesita
   que alguien busque en su computador, el modelo no está implementado: está
   guardado.
2. **«¿Qué NO puede hacer este modelo?»** Un equipo que no tiene lista la
   respuesta no terminó de examinarlo. Las limitaciones se declaran antes de
   que las descubra un reclamo.
3. **«¿Quién decide recalibrar, con qué gatillo y en qué plazo?»** Si el nombre
   no está escrito hoy, el día que el indicador se ponga rojo la respuesta va a
   ser una reunión, no una acción.

### Entrega 3 — miércoles 23 de septiembre

la parte A (40%) + Lab 3 · parte B y proyecto final (60%): notebook reproducible, informe
técnico, model card y resumen ejecutivo. Notas y feedback: miércoles 30.